# Feature Engineering

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv("../data/raw/train.csv")

C:\Users\ASUS TUF\AppData\Local\Temp\ipykernel_18588\513577916.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


## 4.1 Date Features

In [4]:
train["Date"] = pd.to_datetime(train["Date"])

In [5]:
train["Year"] = train["Date"].dt.year

In [6]:
train["Month"] = train["Date"].dt.month

In [7]:
train["Day"] = train["Date"].dt.day

In [8]:
train[["Date", "Year", "Month", "Day"]].head()

,Date,Year,Month,Day
0,2015-07-31,2015,7,31
1,2015-07-31,2015,7,31
2,2015-07-31,2015,7,31
3,2015-07-31,2015,7,31
4,2015-07-31,2015,7,31


### Task Completed
Dateમાંથી Year, Month અને Day features બનાવ્યા.

## 4.2 Week / Month / Year Features

In [9]:
train["WeekOfYear"] = train["Date"].dt.isocalendar().week

In [10]:
train[["Date", "Year", "Month", "WeekOfYear"]].head()

,Date,Year,Month,WeekOfYear
0,2015-07-31,2015,7,31
1,2015-07-31,2015,7,31
2,2015-07-31,2015,7,31
3,2015-07-31,2015,7,31
4,2015-07-31,2015,7,31


### Task Completed
WeekOfYear feature બનાવ્યું.

## 4.3 Store-related Features

In [11]:
store = pd.read_csv("../data/raw/store.csv")

In [12]:
store.shape

(1115, 10)

In [13]:
store.columns

Index(['Store', 'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'],
      dtype='str')

In [14]:
store["Store"].nunique()

1115

In [15]:
store["Store"].duplicated().sum()

np.int64(0)

In [16]:
train = train.merge(store, on="Store", how="left")

In [17]:
train.shape

(1017209, 22)

In [18]:
train.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear',
       'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'],
      dtype='str')

In [19]:
train[["Store", "StoreType", "Assortment", "CompetitionDistance", "Promo2"]].head()

,Store,StoreType,Assortment,CompetitionDistance,Promo2
0,1,c,a,1270.0,0
1,2,a,a,570.0,1
2,3,a,a,14130.0,1
3,4,c,c,620.0,0
4,5,a,a,29910.0,0


### Task Completed
Storeની additional information train data સાથે merge કરી.

## 4.4 Holiday & Promo Features

In [20]:
train["StateHoliday"].value_counts()

StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [21]:
train["StateHoliday"] = train["StateHoliday"].astype(str)

In [22]:
train["StateHoliday"].value_counts()

StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [23]:
train["IsStateHoliday"] = (train["StateHoliday"] != "0").astype(int)

In [24]:
train["IsStateHoliday"].value_counts()

IsStateHoliday
0    986159
1     31050
Name: count, dtype: int64

In [25]:
train["SchoolHoliday"].head()

0    1
1    1
2    1
3    1
4    1
Name: SchoolHoliday, dtype: int64

### Task Completed
StateHoliday અને SchoolHoliday features તૈયાર કર્યા અને Promo પહેલેથી binary feature હોવાથી તેને નવી columnમાં બદલવાની જરૂર નથી.

## 4.5 – Lag Features

In [26]:
train = train.sort_values(["Store", "Date"])

In [27]:
train["Lag_1_Sales"] = train.groupby("Store")["Sales"].shift(1)

In [28]:
train[["Store", "Date", "Sales", "Lag_1_Sales"]].head()

,Store,Date,Sales,Lag_1_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,0.0
1013865,1,2013-01-03,4327,5530.0
1012750,1,2013-01-04,4486,4327.0
1011635,1,2013-01-05,4997,4486.0


### Task Completed
દરેક Store માટે previous day's Sales પરથી `Lag_1_Sales` feature બનાવ્યું.

In [29]:
train["Lag_7_Sales"] = train.groupby("Store")["Sales"].shift(7)

In [30]:
train[["Store", "Date", "Sales", "Lag_1_Sales","Lag_7_Sales" ]].head(10)

,Store,Date,Sales,Lag_1_Sales,Lag_7_Sales
1016095,1,2013-01-01,0,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN
1013865,1,2013-01-03,4327,5530.0,NaN
1012750,1,2013-01-04,4486,4327.0,NaN
1011635,1,2013-01-05,4997,4486.0,NaN
1010520,1,2013-01-06,0,4997.0,NaN
1009405,1,2013-01-07,7176,0.0,NaN
1008290,1,2013-01-08,5580,7176.0,0.0
1007175,1,2013-01-09,5471,5580.0,5530.0
1006060,1,2013-01-10,4892,5471.0,4327.0


### Task Completed
દરેક Store માટે 7 દિવસ પહેલાંની Sales પરથી `Lag_7_Sales` feature બનાવ્યું.

In [31]:
train["Lag_14_Sales"] = train.groupby("Store")["Sales"].shift(14)

In [32]:
train[["Store", "Date", "Sales", "Lag_1_Sales", "Lag_7_Sales", "Lag_14_Sales"]].head(20)

,Store,Date,Sales,Lag_1_Sales,Lag_7_Sales,Lag_14_Sales
1016095,1,2013-01-01,0,NaN,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN,NaN
1013865,1,2013-01-03,4327,5530.0,NaN,NaN
1012750,1,2013-01-04,4486,4327.0,NaN,NaN
1011635,1,2013-01-05,4997,4486.0,NaN,NaN
1010520,1,2013-01-06,0,4997.0,NaN,NaN
1009405,1,2013-01-07,7176,0.0,NaN,NaN
1008290,1,2013-01-08,5580,7176.0,0.0,NaN
1007175,1,2013-01-09,5471,5580.0,5530.0,NaN
1006060,1,2013-01-10,4892,5471.0,4327.0,NaN


### Task Completed
દરેક Store માટે 14 દિવસ પહેલાંની Sales પરથી `Lag_14_Sales` feature બનાવ્યું.

## 4.6 — Rolling Feature

In [33]:
train.groupby("Store")["Sales"].rolling(7).mean()

Store         
1      1016095            NaN
       1014980            NaN
       1013865            NaN
       1012750            NaN
       1011635            NaN
                     ...     
1115   5574       5713.000000
       4459       6144.285714
       3344       6475.571429
       2229       6797.714286
       1114       7206.857143
Name: Sales, Length: 1017209, dtype: float64

In [35]:
train["Rolling_7_Sales"] = train.groupby("Store")["Sales"].rolling(7).mean().reset_index(level=0, drop=True)

In [36]:
train[["Store", "Date", "Sales", "Rolling_7_Sales"]].head(10)

,Store,Date,Sales,Rolling_7_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,NaN
1013865,1,2013-01-03,4327,NaN
1012750,1,2013-01-04,4486,NaN
1011635,1,2013-01-05,4997,NaN
1010520,1,2013-01-06,0,NaN
1009405,1,2013-01-07,7176,3788.000000
1008290,1,2013-01-08,5580,4585.142857
1007175,1,2013-01-09,5471,4576.714286
1006060,1,2013-01-10,4892,4657.428571


### Task Completed
દરેક Store માટે 7 દિવસની average Sales પરથી `Rolling_7_Sales` feature બનાવ્યું.

In [37]:
train["Rolling_14_Sales"] = train.groupby("Store")["Sales"].rolling(14).mean().reset_index(level=0, drop=True)

In [38]:
train[["Store", "Date", "Sales", "Rolling_7_Sales", "Rolling_14_Sales"]].head(20)

,Store,Date,Sales,Rolling_7_Sales,Rolling_14_Sales
1016095,1,2013-01-01,0,NaN,NaN
1014980,1,2013-01-02,5530,NaN,NaN
1013865,1,2013-01-03,4327,NaN,NaN
1012750,1,2013-01-04,4486,NaN,NaN
1011635,1,2013-01-05,4997,NaN,NaN
1010520,1,2013-01-06,0,NaN,NaN
1009405,1,2013-01-07,7176,3788.000000,NaN
1008290,1,2013-01-08,5580,4585.142857,NaN
1007175,1,2013-01-09,5471,4576.714286,NaN
1006060,1,2013-01-10,4892,4657.428571,NaN


In [39]:
train["Rolling_30_Sales"] = train.groupby("Store")["Sales"].rolling(30).mean().reset_index(level=0, drop=True)

In [40]:
train[["Store", "Date", "Sales", "Rolling_7_Sales", "Rolling_14_Sales", "Rolling_30_Sales"]].head(35)

,Store,Date,Sales,Rolling_7_Sales,Rolling_14_Sales,Rolling_30_Sales
1016095,1,2013-01-01,0,NaN,NaN,NaN
1014980,1,2013-01-02,5530,NaN,NaN,NaN
1013865,1,2013-01-03,4327,NaN,NaN,NaN
1012750,1,2013-01-04,4486,NaN,NaN,NaN
1011635,1,2013-01-05,4997,NaN,NaN,NaN
1010520,1,2013-01-06,0,NaN,NaN,NaN
1009405,1,2013-01-07,7176,3788.000000,NaN,NaN
1008290,1,2013-01-08,5580,4585.142857,NaN,NaN
1007175,1,2013-01-09,5471,4576.714286,NaN,NaN
1006060,1,2013-01-10,4892,4657.428571,NaN,NaN


### Task Completed
દરેક Store માટે 7, 14 અને 30 દિવસની average Sales પરથી `Rolling_7_Sales`, `Rolling_14_Sales` અને `Rolling_30_Sales` features બનાવ્યા.

## 4.7 — Feature Leakage Check

In [44]:
train["Rolling_7_Sales"] = train.groupby("Store")["Sales"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

In [45]:
train[["Store", "Date", "Sales", "Rolling_7_Sales"]].head(10)

,Store,Date,Sales,Rolling_7_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,NaN
1013865,1,2013-01-03,4327,NaN
1012750,1,2013-01-04,4486,NaN
1011635,1,2013-01-05,4997,NaN
1010520,1,2013-01-06,0,NaN
1009405,1,2013-01-07,7176,NaN
1008290,1,2013-01-08,5580,3788.000000
1007175,1,2013-01-09,5471,4585.142857
1006060,1,2013-01-10,4892,4576.714286


In [46]:
train["Rolling_14_Sales"] = train.groupby("Store")["Sales"].transform(
    lambda x: x.shift(1).rolling(14).mean()
)

In [47]:
train[["Store", "Date", "Sales", "Rolling_14_Sales"]].head(20)


,Store,Date,Sales,Rolling_14_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,NaN
1013865,1,2013-01-03,4327,NaN
1012750,1,2013-01-04,4486,NaN
1011635,1,2013-01-05,4997,NaN
1010520,1,2013-01-06,0,NaN
1009405,1,2013-01-07,7176,NaN
1008290,1,2013-01-08,5580,NaN
1007175,1,2013-01-09,5471,NaN
1006060,1,2013-01-10,4892,NaN


In [48]:
train["Rolling_30_Sales"] = train.groupby("Store")["Sales"].transform(
    lambda x: x.shift(1).rolling(30).mean()
)

In [49]:
train[["Store", "Date", "Sales", "Rolling_30_Sales"]].head(35)

,Store,Date,Sales,Rolling_30_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,NaN
1013865,1,2013-01-03,4327,NaN
1012750,1,2013-01-04,4486,NaN
1011635,1,2013-01-05,4997,NaN
1010520,1,2013-01-06,0,NaN
1009405,1,2013-01-07,7176,NaN
1008290,1,2013-01-08,5580,NaN
1007175,1,2013-01-09,5471,NaN
1006060,1,2013-01-10,4892,NaN


### Task Completed
દરેક Store માટે પાછલા 30 દિવસની average Sales પરથી `Rolling_30_Sales` feature બનાવ્યું.

## 4.7 — Feature Leakage Check

In [50]:
train[[
    "Store",
    "Date",
    "Sales",
    "Lag_1_Sales",
    "Lag_7_Sales",
    "Lag_14_Sales",
    "Rolling_7_Sales",
    "Rolling_14_Sales",
    "Rolling_30_Sales"
]].head(35)

,Store,Date,Sales,Lag_1_Sales,Lag_7_Sales,Lag_14_Sales,Rolling_7_Sales,Rolling_14_Sales,Rolling_30_Sales
1016095,1,2013-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN,NaN,NaN,NaN,NaN
1013865,1,2013-01-03,4327,5530.0,NaN,NaN,NaN,NaN,NaN
1012750,1,2013-01-04,4486,4327.0,NaN,NaN,NaN,NaN,NaN
1011635,1,2013-01-05,4997,4486.0,NaN,NaN,NaN,NaN,NaN
1010520,1,2013-01-06,0,4997.0,NaN,NaN,NaN,NaN,NaN
1009405,1,2013-01-07,7176,0.0,NaN,NaN,NaN,NaN,NaN
1008290,1,2013-01-08,5580,7176.0,0.0,NaN,3788.000000,NaN,NaN
1007175,1,2013-01-09,5471,5580.0,5530.0,NaN,4585.142857,NaN,NaN
1006060,1,2013-01-10,4892,5471.0,4327.0,NaN,4576.714286,NaN,NaN


In [51]:
train[[
    "Sales",
    "Lag_1_Sales",
    "Lag_7_Sales",
    "Lag_14_Sales",
    "Rolling_7_Sales",
    "Rolling_14_Sales",
    "Rolling_30_Sales"
]].isnull().sum()

Sales                   0
Lag_1_Sales          1115
Lag_7_Sales          7805
Lag_14_Sales        15610
Rolling_7_Sales      7805
Rolling_14_Sales    15610
Rolling_30_Sales    33450
dtype: int64

### Task Completed
Lag અને Rolling featuresમાં current અથવા future Salesનો ઉપયોગ થતો નથી તે verify કર્યું. શરૂઆતની NaN values historical data ઉપલબ્ધ ન હોવાથી expected છે.

##  4.8 — Final Feature Verification

In [52]:
feature_columns = [
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2",
    "IsStateHoliday",
    "Lag_1_Sales",
    "Lag_7_Sales",
    "Lag_14_Sales",
    "Rolling_7_Sales",
    "Rolling_14_Sales",
    "Rolling_30_Sales"
]

print("Total Features:", len(feature_columns))
print("\nFeatures:")
print(feature_columns)

Total Features: 15

Features:
['Year', 'Month', 'Day', 'WeekOfYear', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'IsStateHoliday', 'Lag_1_Sales', 'Lag_7_Sales', 'Lag_14_Sales', 'Rolling_7_Sales', 'Rolling_14_Sales', 'Rolling_30_Sales']


In [53]:
for col in feature_columns:
    print(col, col in train.columns)

Year True
Month True
Day True
WeekOfYear True
StoreType True
Assortment True
CompetitionDistance True
Promo2 True
IsStateHoliday True
Lag_1_Sales True
Lag_7_Sales True
Lag_14_Sales True
Rolling_7_Sales True
Rolling_14_Sales True
Rolling_30_Sales True


### Task Completed
બધા engineered features `train` datasetમાં યોગ્ય રીતે હાજર છે તે verify કર્યું.